In [ ]:
from pathlib import Path

import tiktoken
import torch

from config import HF_MODELS, INSTRUCTION_DATA_DIR, MODEL_CONFIG, VARIANT
from data.dataset import data_split, get_instruction_loaders
from inference.load_weights import load_from_hf
from model.gpt import GPTModel
from finetune.instructure_follower_finetuning import loading_model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Loading the model

In [ ]:
model , config = load_from_hf(VARIANT)
param_count: int = sum(p.numel() for p in model.parameters())
print(f"Parameters: {param_count:,}")

In [ ]:
context_size: int = config["context_length"]  # e.g. 256 (tiny) or 1024 (medium)
# train_loader, val_loader, _ = split_and_get_loaders(REMOTE_DATA) #add the function to get the data

Adding a classification head

In [ ]:
# freeze the model
for param in model.parameters():
   param.requires_grad = False

In [ ]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
in_features=BASE_CONFIG["emb_dim"], 
   out_features=num_classes
)

In [ ]:
# Make the final LayerNorm and last transformer block trainable
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape) # shape: (batch_size, num_tokens)

In [ ]:
with torch.no_grad():
   outputs = model(inputs)
print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)
print("Last output token:", outputs[:, -1, :])  

Implementing the utility function to calculate the classification loss and accuracy of the model to evaluate the model's performance

In [ ]:
# Obtain the class label
probas = torch.softmax(outputs[:, -1, :], dim=-1)
label = torch.argmax(probas)
print("Class label:", label.item())

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
   model.eval()
   correct_predictions, num_examples = 0, 0
   if num_batches is None:
       num_batches = len(data_loader)
   else:
       num_batches = min(num_batches, len(data_loader))
   for i, (input_batch, target_batch) in enumerate(data_loader):
       if i < num_batches:
           input_batch, target_batch = input_batch.to(device), target_batch.to(device)
           with torch.no_grad():
               logits = model(input_batch)[:, -1, :]             #A
           predicted_labels = torch.argmax(logits, dim=-1)
           num_examples += predicted_labels.shape[0]
           correct_predictions += (predicted_labels == target_batch).sum().item()
       else:
           break
   return correct_predictions / num_examples